In [ ]:
MODEL_CODE = "knn"
MODEL_NAME = "KNN"

# 06: KNN Modeli 

**Proje:** Framingham Kalp Çalışması verisi ile inme (stroke) tahmini · **Model ailesi:** mesafe / sınır tabanlı

**KNN (K-En Yakın Komşu)** yeni bir hastaya en çok benzeyen *k* hastaya bakar; onların ağırlıklı çoğunluğu inme geçirdiyse "inme var" der. Eğitim aşaması yoktur, tüm karar mesafe hesabına dayanır. Bu yüzden ölçeklemeye ve gereksiz sütunlara çok duyarlıdır.

**Bu notebook'ta yapılanlar:** hiperparametre araması · k'nın overfitting'e etkisi · dengeleme yöntemi karşılaştırması · feature engineering etkisi · CV değerlendirmesi ve eşik seçimi · öğrenme eğrisi · test seti · XAI

## 1. Kurulum

In [ ]:
# shap and lime are not preinstalled on Colab
try:
    import lime
    import shap
except ImportError:
    %pip install -q shap lime

In [ ]:
import sys
from pathlib import Path

# Repo: ../src. Colab: the src/*.py files uploaded to the working directory.
SRC_DIR = next((p for p in (Path("../src"), Path("src"), Path(".")) if (p / "config.py").exists()), None)
if SRC_DIR is None:
    raise FileNotFoundError("config.py bulunamadı. Colab'da src klasöründeki .py dosyalarını sol menüden yükleyin.")
sys.path.insert(0, str(SRC_DIR.resolve()))

# Çıktı klasörleri: repo kökünde models/ ve figures/ (Colab'da çalışma klasöründe)
ROOT = SRC_DIR.resolve().parent if SRC_DIR.name == "src" else Path(".")
MODEL_DIR = ROOT / "models";  MODEL_DIR.mkdir(exist_ok=True)
FIG_DIR = ROOT / "figures";   FIG_DIR.mkdir(exist_ok=True)

import time
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

from config import MAIN_METRIC, RANDOM_STATE, TARGET_RECALL
from preprocessing import build_pipeline, load_data
from metrics import (SCORING, cv_evaluate, learning_curve_analysis, shared_cv, test_evaluate,
                     threshold_analysis)
from explain import explain

warnings.filterwarnings("ignore")

## 2. Veri

 `train.csv` / `test.csv`. Ön işleme (feature engineering, encoding, ölçekleme, SMOTE) `build_pipeline()` içinde yapılır; test seti en sona kadar kullanılmaz.

In [ ]:
X_train, X_test, y_train, y_test = load_data()
print("Eğitim:", X_train.shape, f"| inme: {y_train.sum()} kişi ({y_train.mean():.2%})")
print("Test:  ", X_test.shape, f"| inme: {y_test.sum()} kişi ({y_test.mean():.2%})")

## 3. Hiperparametre araması

| Parametre | Değerler | Anlamı |
|---|---|---|
| `n_neighbors` (k) | 5, 11, 21, 31, 51, 75 | Kaç komşuya bakılacağı. Küçük k gürültüye duyarlı (overfitting), büyük k daha düzgün ama ayrıntıyı kaçırabilir. Tek sayı: oylamada eşitlik olmaz |
| `weights` | uniform, distance | Tüm komşular eşit mi, yakın olanın oyu daha mı ağır |
| `p` | 1, 2 | Mesafe: 1 = Manhattan, 2 = Öklid |

Ortak katlar (`shared_cv()`) ve ortak metrikler (`SCORING`) kullanılır; en iyi model `MAIN_METRIC` (PR-AUC) ile seçilir.

In [ ]:
def search_table(search, params):
    """Hiperparametre denemeleri: eğitim ve CV skoru, aradaki fark (overfitting göstergesi)."""
    r = pd.DataFrame(search.cv_results_)
    t = r[[f"param_{p}" for p in params]].copy()
    t.columns = [p.replace("model__", "") for p in params]
    for m in [MAIN_METRIC, "recall", "roc_auc"]:
        t[f"{m}_cv"] = r[f"mean_test_{m}"]
    t[f"{MAIN_METRIC}_train"] = r[f"mean_train_{MAIN_METRIC}"]
    t["fark (train - cv)"] = t[f"{MAIN_METRIC}_train"] - t[f"{MAIN_METRIC}_cv"]
    return t.sort_values(f"{MAIN_METRIC}_cv", ascending=False).round(4)

In [ ]:
params = {
    "model__n_neighbors": [5, 11, 21, 31, 51, 75],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2],
}
start = time.time()
search = GridSearchCV(build_pipeline(KNeighborsClassifier(), balancing="smote"), params,
                      cv=shared_cv(), scoring=SCORING, refit=MAIN_METRIC,
                      n_jobs=-1, return_train_score=True)
search.fit(X_train, y_train)
best = search.best_estimator_

print(f"Süre: {time.time() - start:.0f} sn")
print("En iyi parametreler:", search.best_params_)
print(f"En iyi CV {MAIN_METRIC}: {search.best_score_:.4f}")
search_table(search, list(params)).head(10)

### k'nın etkisi (overfitting)

`weights = distance` seçildiğinde her eğitim noktası kendi en yakın komşusudur (mesafe 0), bu yüzden eğitim skoru neredeyse her zaman 1 çıkar: KNN eğitim verisini ezberler. Asıl önemli olan CV skorudur. k büyüdükçe eğitim ve CV skorları arasındaki fark kapanır.

In [ ]:
r = pd.DataFrame(search.cv_results_)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, w in zip(axes, ["uniform", "distance"]):
    sub = r[(r["param_model__weights"] == w) & (r["param_model__p"] == search.best_params_["model__p"])]
    ax.plot(sub["param_model__n_neighbors"], sub[f"mean_train_{MAIN_METRIC}"], "o-", label="eğitim")
    ax.plot(sub["param_model__n_neighbors"], sub[f"mean_test_{MAIN_METRIC}"], "o-", label="CV")
    ax.set_title(f"KNN, weights = {w}")
    ax.set_xlabel("k (komşu sayısı)")
    ax.legend()
axes[0].set_ylabel(MAIN_METRIC)
plt.tight_layout()
plt.savefig(FIG_DIR / f"{MODEL_CODE}_k_overfitting.png", dpi=200)
plt.show()

## 4. Dengeleme yöntemi

KNN'de `class_weight` parametresi yoktur. Bu yüzden iki seçenek karşılaştırılır: **SMOTE** ve **dengeleme yok**. Dengeleme olmadan KNN'in komşularının çoğu "inme yok" sınıfından geleceği için recall'un düşmesi beklenir.

In [ ]:
best_params = {k.replace("model__", ""): v for k, v in search.best_params_.items()}
rows = {}
for balancing in ["smote", "none"]:
    res = cv_evaluate(build_pipeline(KNeighborsClassifier(**best_params), balancing=balancing),
                      X_train, y_train, model_name=f"KNN ({balancing})", save=False, verbose=False)
    rows[balancing] = res["summary"]["cv_mean"]
pd.DataFrame(rows).T.round(3)

## 5. Feature engineering'in etkisi

Ortak pipeline 5 yeni özellik ekler (17 → 28 sütun). Bunların çoğu tansiyondan türetildiği için birbirine çok benzer. Mesafe tabanlı modellerde gereksiz sütunlar mesafeyi bozabilir. En iyi parametrelerle feature engineering açık ve kapalı karşılaştırılır (bu deneyler `results.csv`'ye yazılmaz).

In [ ]:
rows = {}
for fe in [True, False]:
    res = cv_evaluate(build_pipeline(KNeighborsClassifier(**best_params), balancing="smote",
                                     feature_engineering=fe),
                      X_train, y_train, model_name=f"KNN (FE={fe})", save=False, verbose=False)
    rows["FE açık" if fe else "FE kapalı"] = res["summary"]["cv_mean"]
pd.DataFrame(rows).T.round(3)

## 6. Final model: CV değerlendirmesi

Seçilen model ortak `cv_evaluate` ile değerlendirilir ve sonucu `results.csv`'ye **bir satır** olarak eklenir. Karşılaştırma notebook'u bu satırı kullanır. Eğitim ve CV skorları arasındaki fark (`gap`) overfitting göstergesidir.

In [ ]:
cv_result = cv_evaluate(best, X_train, y_train, model_name=MODEL_NAME,
                        note=f"SMOTE + KNN, GridSearch: {best_params}")

## 7. Eşik seçimi

Varsayılan eşik 0.5'tir, ama inmeyi kaçırmak yanlış alarmdan pahalı olduğu için eşik, **test setine bakılmadan** CV tahminleri (OOF) üzerinde seçilir: hedef recall'u (%80) en az yanlış alarmla sağlayan eşik.

In [ ]:
table, threshold = threshold_analysis(cv_result["y"], cv_result["oof_score"], model_name=MODEL_NAME)

## 8. Overfitting kontrolü: öğrenme eğrisi

Eğitim verisi büyüdükçe eğitim ve CV skorları birbirine yaklaşıyorsa model genelleşiyor demektir. Büyük ve kapanmayan bir fark overfitting, ikisinin birlikte düşük kalması underfitting gösterir.

In [ ]:
lc = learning_curve_analysis(best, X_train, y_train, model_name=MODEL_NAME)
lc.round(3)

## 9. Test seti (yalnızca bir kez)

Final model tüm eğitim verisiyle yeniden eğitilir, seçilen eşikle test setinde değerlendirilir ve `models/` klasörüne kaydedilir.

In [ ]:
best.fit(X_train, y_train)
test_result = test_evaluate(best, X_test, y_test, threshold=threshold,
                            model_name=MODEL_NAME, cv_result=cv_result)

joblib.dump(best, MODEL_DIR / f"{MODEL_CODE}.joblib")
print(f"Model kaydedildi: models/{MODEL_CODE}.joblib")

## 10. XAI: modelin açıklanması

KNN iç yapısı doğrudan okunamayan bir model olduğu için ortak `explain()` fonksiyonu **KernelExplainer** kullanır. Bu yöntem yavaş olduğundan test setinden 100 hasta, eğitim verisinin 20 küme merkezi ve hasta başına 300 özellik kombinasyonu kullanılır (yaklaşık 1 ila 3 dakika). XAI fonksiyonuna aramanın kendisi değil, `search.best_estimator_` (`best`) verilir.

**Grafik okuma:** Bar grafiğinde çubuk uzunluğu önemi gösterir. Özet grafiğinde kırmızı = özelliğin değeri yüksek, mavi = düşük; nokta sağdaysa inme riskini artırıyor. LIME'da yeşil çubuklar "inme var" kararını destekleyen koşullardır.

In [ ]:
# explain() grafikleri sırayla gösterir: SHAP bar, SHAP özet, LIME.
# Her grafik gösterilirken figures/ klasörüne de kaydedilir.
_names = iter([f"{MODEL_CODE}_shap_bar.png", f"{MODEL_CODE}_shap_summary.png", f"{MODEL_CODE}_lime.png"])
_show = plt.show
def _save_and_show(*args, **kwargs):
    name = next(_names, None)
    if name:
        plt.gcf().savefig(FIG_DIR / name, dpi=200, bbox_inches="tight")
    _show(*args, **kwargs)

plt.show = _save_and_show
try:
    start = time.time()
    xai = explain(best, X_train, X_test, model_name=MODEL_NAME,
                  sample_size=100, background_size=20, kernel_nsamples=300)
    print(f"\nXAI süresi: {time.time() - start:.0f} sn")
finally:
    plt.show = _show
print("Grafikler kaydedildi:", [p.name for p in sorted(FIG_DIR.glob(f"{MODEL_CODE}_*.png"))])

In [ ]:
xai["shap_importance"].head(10)